In [47]:
import torch
import numpy as np
from main import ModelHolder, get_config
from dataloader import d2v_pairer

class TrainedModel():
    def __init__(self, model_file_path, config_file_path):
        state_dict = torch.load(model_file_path)
        self.model = ModelHolder(cfg_all=get_config(cfg_file=config_file_path))
        self.model.load_state_dict(state_dict['model_state_dict'])
        self.embed_meta = None
        self.pos_enc = None

    def fit(self, xs_meta, ys_meta):
        pass

    def predict(self, xs_target):
        if self.embed_meta is None or self.pos_enc is None:
            raise Exception("Model has not been fitted yet!")
        
        xs_target = xs_target.unsqueeze(0)
        with torch.no_grad():
            ys_pred_target = self.model.forward_target(xs_target, self.embed_meta, self.pos_enc)

        return torch.argmax(ys_pred_target.view(-1, 2), dim=1).numpy()

class FLAT(TrainedModel):
    def __init__(self, model_file_path, config_file_path):
        super().__init__(model_file_path, config_file_path)

    def fit(self, xs_meta, ys_meta):
        xs_meta, ys_meta = xs_meta.unsqueeze(0), ys_meta.unsqueeze(0)

        pairs_meta = d2v_pairer(xs_meta, ys_meta)
        with torch.no_grad():
            self.embed_meta, self.pos_enc = self.model.forward_meta(pairs_meta)

    def __repr__(self):
        return "FLAT"
    
class FLATadapt(TrainedModel):
    def __init__(self, model_file_path, config_file_path):
        super().__init__(model_file_path, config_file_path)

    def fit(self, xs_meta, ys_meta):
        xs_meta, ys_meta = xs_meta.unsqueeze(0), ys_meta.unsqueeze(0)
        pairs_meta = d2v_pairer(xs_meta, ys_meta)
        with torch.no_grad():
            embed_meta, pos_enc = self.model.forward_meta(pairs_meta)

        embed_meta.requires_grad = True
        pos_enc.requires_grad = True
        optim_pos = torch.optim.Adam([pos_enc], lr=0.001)
        optim_embed = torch.optim.Adam([embed_meta], lr=0.075)
        for _ in range(5):
            # Make predictions on meta set and calc loss
            preds = self.model.forward_target(xs_meta, embed_meta, pos_enc)
            loss = torch.nn.functional.cross_entropy(preds.squeeze(), ys_meta.long().squeeze())
            loss.backward()
            optim_pos.step()
            optim_embed.step()
            optim_embed.zero_grad()
            optim_pos.zero_grad()

        self.embed_meta = embed_meta
        self.pos_enc = pos_enc

    def __repr__(self):
        return "FLATadapt"

In [52]:
from sklearn.datasets import load_breast_cancer
import pandas as pd

# load dataset
data = load_breast_cancer()
X, y = data.data, data.target
df = pd.DataFrame(X)
df['y'] = y

num_meta = 5
num_target = 5

# sample meta and target data
subsample_df = df.sample(num_meta + num_target)

# standardize features
subsample_xs = subsample_df.drop('y', axis=1).values
subsample_xs = (subsample_xs - subsample_xs.mean(axis=0)) / subsample_xs.std(axis=0)

xs_meta, xs_target = subsample_xs[:num_meta], subsample_xs[num_meta:]
ys_meta, ys_target = subsample_df['y'].values[:num_meta], subsample_df['y'].values[num_meta:]

# initialize FLAT models
model_file_path = '../model-5.pt'
config_file_path = '../config.toml'

flat = FLAT(model_file_path=model_file_path, config_file_path=config_file_path)
flatadapt = FLATadapt(model_file_path=model_file_path, config_file_path=config_file_path)

# fit FLAT to the meta data
flat.fit(
    xs_meta=torch.tensor(xs_meta, dtype=torch.float32),
    ys_meta=torch.tensor(ys_meta, dtype=torch.long)
)
flatadapt.fit(
    xs_meta=torch.tensor(xs_meta, dtype=torch.float32),
    ys_meta=torch.tensor(ys_meta, dtype=torch.long)
)

# predict on the target data
ys_pred_target_flat = flat.predict(
    xs_target=torch.tensor(xs_target, dtype=torch.float32)
)
ys_pred_target_flatadapt = flatadapt.predict(
    xs_target=torch.tensor(xs_target, dtype=torch.float32)
)

# print results
print("======FLAT========")
print("Predicted labels: ", ys_pred_target_flat)
print("True labels: ", ys_target)
print("Accuracy: ", np.mean(ys_pred_target_flat == ys_target))

print("======FLATadapt========")
print("Predicted labels: ", ys_pred_target_flatadapt)
print("True labels: ", ys_target)
print("Accuracy: ", np.mean(ys_pred_target_flatadapt == ys_target))

======FLAT========
Predicted labels:  [0 1 0 0 1]
True labels:  [0 1 1 0 1]
Accuracy:  0.8
======FLATadapt========
Predicted labels:  [0 1 1 0 1]
True labels:  [0 1 1 0 1]
Accuracy:  1.0
